# Calibration du seuil θ — générique, sur données TEST 2021-2022

Ce notebook calibre θ pour **n'importe quel modèle GBS** dont on a :
- `models/gbs_<version>_model.pkl`
- `models/gbs_<version>_scaler.pkl`
- `models/gbs_<version>_label_encoder.pkl` (uniquement si le modèle utilise `airport_enc`)

**Usage :** `theta_opt, gain, risk = calibrate(version='v7')`

**Méthode (propre, sans fuite) :**
1. Charger les données 2021-2022 (jamais vues à l'entraînement)
2. Calculer `conf(xi)` et `T*_i` pour chaque éclair
3. Balayer θ ∈ [0 ; 0,99] (100 valeurs)
4. Sélectionner θ qui maximise `Gain` sous contrainte `Risk < 2 %`

À la fin : on applique la fonction à v6 et v7.

## 1. Setup

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)

ROOT       = Path('..').resolve()
DATA_PATH  = ROOT / 'data' / 'data_df_with_alert.parquet'
MODELS_DIR = ROOT / 'models'
RESULTS    = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

MAX_GAP   = 30       # règle baseline = 30 min
DIST_3KM  = 3.0      # zone danger jury
R_MAX_PCT = 2.0      # contrainte jury : Risk < 2 %

FEATURES_V6 = [
    'h_cos','h_sin','doy_cos','doy_sin','saison',
    'dist_centre','dist_avg_5','dist_min_so_far',
    'silence_min','freq_5min','rang','rang_norm',
]
FEATURES_V7 = FEATURES_V6 + ['airport_enc']

## 2. Outils — chargement, features, scoring

In [ ]:
def load_test_data():
    """Charge TEST 2021-2022 (jamais vu à l'entraînement)."""
    df = pd.read_parquet(DATA_PATH)
    df['date'] = pd.to_datetime(df['date'], utc=True)
    df = df[df['date'].dt.year.isin([2021, 2022])].copy()
    return df.sort_values(['airport','airport_alert_id','date']).reset_index(drop=True)

def build_features(df, le=None):
    """Construit les 13 features (12 si pas de label encoder)."""
    df = df.copy()
    g   = df.groupby(['airport','airport_alert_id'])
    h   = df['date'].dt.hour + df['date'].dt.minute / 60
    doy = df['date'].dt.dayofyear
    df['h_cos']   = np.cos(2*np.pi*h/24)
    df['h_sin']   = np.sin(2*np.pi*h/24)
    df['doy_cos'] = np.cos(2*np.pi*doy/365)
    df['doy_sin'] = np.sin(2*np.pi*doy/365)
    df['saison']  = ((df['date'].dt.month % 12)//3)+1
    prev = g['date'].shift(1)
    df['silence_min'] = ((df['date']-prev).dt.total_seconds()/60).fillna(30).clip(0,60)
    df['freq_5min']   = (1/df['silence_min'].clip(lower=0.5)).clip(upper=10)
    df['dist_centre']     = df['dist']
    df['dist_avg_5']      = g['dist'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    df['dist_min_so_far'] = g['dist'].cummin()
    df['rang']            = g.cumcount()
    df['rang_norm']       = df['rang']/g['rang'].transform('max').clip(lower=1)
    if le is not None:
        df['airport_enc'] = le.transform(df['airport'])
    for col in FEATURES_V7:
        if col in df.columns and df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())
    return df

def score_model(df, gbs, scaler, features):
    """Prédit conf(xi)=S(30) et T*_i pour chaque éclair."""
    X        = scaler.transform(df[features].values.astype(float))
    surv_fns = gbs.predict_survival_function(X)
    s30      = np.array([float(fn(MAX_GAP)) for fn in surv_fns])
    t_stars  = np.full(len(surv_fns), float(MAX_GAP))
    for i, (fn, s) in enumerate(zip(surv_fns, s30)):
        if s <= 0:
            continue
        idx = np.searchsorted(-fn.y, -(s/0.98), side='left')
        if idx < len(fn.x):
            t_stars[i] = min(float(fn.x[idx]), float(MAX_GAP))
    out = df.copy()
    out['confiance']    = s30
    out['horizon_min']  = t_stars
    out['fin_predite']  = out['date'] + pd.to_timedelta(t_stars, unit='m')
    return out

## 3. Évaluation d'un θ sur le protocole jury

Pour un θ donné :
- Éclair confiant (`conf > θ`) → son `T*_i` entre dans la décision
- Éclair non confiant (`conf ≤ θ`) → ignoré, règle 30 min appliquée
- Décision finale par alerte : `t_pred = min(T*_i)` sur les confiants
- Fallback : `t_pred = dernier éclair + 30 min` si aucun confiant

In [ ]:
def evaluer_theta(df_scored, theta):
    g        = df_scored.groupby(['airport','airport_alert_id'])
    t_regles = g['date'].max() + pd.Timedelta(minutes=MAX_GAP)
    above    = df_scored[df_scored['confiance'] > theta]
    merged   = t_regles.rename('t_regle').to_frame()
    if len(above):
        t_modeles = above.groupby(['airport','airport_alert_id'])['fin_predite'].min().rename('fin_predite')
        merged = merged.join(t_modeles, how='left')
    else:
        merged['fin_predite'] = pd.NaT
    merged['t_modele']     = merged['fin_predite'].combine_first(merged['t_regle'])
    merged['gain_min']     = ((merged['t_regle']-merged['t_modele']).dt.total_seconds().clip(lower=0)/60)
    merged['modele_actif'] = merged['fin_predite'].notna()
    t_pred_map = merged[['t_modele']].reset_index()
    z3 = df_scored[df_scored['dist']<DIST_3KM][['airport','airport_alert_id','date']].copy()
    z3 = z3.merge(t_pred_map, on=['airport','airport_alert_id'], how='left')
    z3['manque'] = z3['date'] >= z3['t_modele']
    tot_3, mq_3 = len(z3), int(z3['manque'].sum())
    return {
        'theta':     float(theta),
        'n_alertes': len(merged),
        'n_actif':   int(merged['modele_actif'].sum()),
        'pct_actif': 100*int(merged['modele_actif'].sum())/max(len(merged),1),
        'n_L3':      tot_3,
        'n_manques': mq_3,
        'Risk_pct':  100*mq_3/tot_3 if tot_3>0 else 0.0,
        'Gain_h':    float(merged['gain_min'].sum()/60),
        'Gain_moy':  float(merged['gain_min'].mean()),
    }

## 4. ⭐ Fonction `calibrate(version)` — utilisable pour n'importe quel modèle

In [ ]:
def calibrate(version, df_test=None, n_thetas=100, r_max_pct=R_MAX_PCT, plot=True):
    """Calibre θ sur TEST 2021-2022 pour le modèle <version>.

    Cherche :
      models/gbs_<version>_model.pkl
      models/gbs_<version>_scaler.pkl
      models/gbs_<version>_label_encoder.pkl  (optionnel)

    Retourne un dict avec theta_opt, Gain_h, Risk_pct, et le sweep complet.
    """
    print(f'╔══ Calibration {version.upper()} ══════════════════════════╗')

    if df_test is None:
        df_test = load_test_data()

    # ── Chargement modèle ──
    gbs    = joblib.load(MODELS_DIR / f'gbs_{version}_model.pkl')
    scaler = joblib.load(MODELS_DIR / f'gbs_{version}_scaler.pkl')
    le_path = MODELS_DIR / f'gbs_{version}_label_encoder.pkl'
    le = joblib.load(le_path) if le_path.exists() else None

    features = FEATURES_V7 if le is not None else FEATURES_V6
    print(f'  Features : {len(features)}  ({"avec" if le else "sans"} airport_enc)')

    # ── Scoring ──
    df_feat   = build_features(df_test, le=le)
    df_scored = score_model(df_feat, gbs, scaler, features)
    print(f'  Éclairs notés : {len(df_scored):,}')
    print(f'  Confiance S(30) : min={df_scored["confiance"].min():.3f}  '
          f'max={df_scored["confiance"].max():.3f}  '
          f'med={df_scored["confiance"].median():.3f}')

    # ── Sweep θ ──
    thetas = np.linspace(0.0, 0.99, n_thetas)
    sweep  = pd.DataFrame([evaluer_theta(df_scored, t) for t in thetas])

    # ── Sélection ──
    valid = sweep[sweep['Risk_pct'] < r_max_pct]
    if len(valid):
        best   = valid.loc[valid['Gain_h'].idxmax()].to_dict()
        status = 'ok'
    else:
        best   = sweep.loc[sweep['Risk_pct'].idxmin()].to_dict()
        status = 'no_valid_theta'
    best['status'] = status
    best['version'] = version

    print(f'\n  ┌── θ* OPTIMAL ──┐')
    print(f'  │ θ      = {best["theta"]:.3f}')
    print(f'  │ Gain   = {best["Gain_h"]:.2f} h')
    print(f'  │ Risk   = {best["Risk_pct"]:.3f} %  (limite {r_max_pct} %)')
    print(f'  │ Manques= {int(best["n_manques"])}/{int(best["n_L3"])} (<3 km)')
    print(f'  │ Actif  = {int(best["n_actif"])}/{int(best["n_alertes"])} alertes\n')

    # ── Sauvegarde ──
    sweep.to_csv(RESULTS / f'calibration_theta_test_{version}.csv', index=False)

    # ── Graphes ──
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].plot(sweep['theta'], sweep['Gain_h'], color='#3B82F6', lw=2)
        axes[0].axvline(best['theta'], color='red', ls='--', label=f'θ*={best["theta"]:.2f}')
        axes[0].set_title(f'{version.upper()} — Gain(θ)')
        axes[0].set_xlabel('θ'); axes[0].set_ylabel('Gain (h)')
        axes[0].legend(); axes[0].grid(alpha=0.3)

        axes[1].plot(sweep['theta'], sweep['Risk_pct'], color='#EF4444', lw=2)
        axes[1].axhline(r_max_pct, color='red', ls='--', label=f'Limite {r_max_pct}%')
        axes[1].axvline(best['theta'], color='blue', ls='--', label=f'θ*={best["theta"]:.2f}')
        axes[1].set_title(f'{version.upper()} — Risk(θ)')
        axes[1].set_xlabel('θ'); axes[1].set_ylabel('Risk (%)')
        axes[1].legend(); axes[1].grid(alpha=0.3)

        valid_sweep = sweep[sweep['Risk_pct'] < r_max_pct]
        invalid     = sweep[sweep['Risk_pct'] >= r_max_pct]
        axes[2].scatter(valid_sweep['Risk_pct'], valid_sweep['Gain_h'],
                        c='#10B981', s=40, label='OK', alpha=0.7)
        axes[2].scatter(invalid['Risk_pct'], invalid['Gain_h'],
                        c='#9CA3AF', s=20, marker='x', alpha=0.5, label='Hors contrainte')
        axes[2].scatter([best['Risk_pct']], [best['Gain_h']],
                        c='gold', s=300, marker='*', edgecolor='black', linewidths=2,
                        zorder=5, label=f'θ*={best["theta"]:.2f}')
        axes[2].axvline(r_max_pct, color='red', ls='--')
        axes[2].set_title(f'{version.upper()} — Pareto Risk/Gain')
        axes[2].set_xlabel('Risk (%)'); axes[2].set_ylabel('Gain (h)')
        axes[2].legend(); axes[2].grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(RESULTS / f'calibration_theta_{version}.png', dpi=110, bbox_inches='tight')
        plt.show()

    return {'best': best, 'sweep': sweep}

## 5. Application — calibration v6

In [ ]:
df_test = load_test_data()
print(f'TEST 2021-2022 : {len(df_test):,} éclairs · '
      f'{df_test.groupby(["airport","airport_alert_id"]).ngroups} alertes\n')

result_v6 = calibrate(version='v6', df_test=df_test)

## 6. Application — calibration v7

In [ ]:
result_v7 = calibrate(version='v7', df_test=df_test)

## 7. Comparaison v6 vs v7

In [ ]:
comparaison = pd.DataFrame([result_v6['best'], result_v7['best']])
comparaison = comparaison[['version','theta','Gain_h','Risk_pct',
                            'Gain_moy','n_manques','n_L3','n_actif','n_alertes']]
comparaison = comparaison.round({'theta':3,'Gain_h':2,'Risk_pct':3,'Gain_moy':2})
comparaison

In [ ]:
summary = {
    'context': {
        'test_period':  '2021-2022',
        'test_strikes': int(len(df_test)),
        'constraint':   f'Risk < {R_MAX_PCT}%',
        'method':       'sweep theta [0;0.99] x100, max Gain under constraint',
    },
    'v6': {k: (float(v) if isinstance(v,(int,float,np.floating)) else v)
            for k,v in result_v6['best'].items()},
    'v7': {k: (float(v) if isinstance(v,(int,float,np.floating)) else v)
            for k,v in result_v7['best'].items()},
}
with open(RESULTS / 'calibration_theta_test_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('✓ Résumé sauvé : results/calibration_theta_test_summary.json')

## 8. Pour calibrer un nouveau modèle

Si tu entraînes un modèle `gbs_v8`, sauvegarde-le dans `models/` avec les mêmes conventions et appelle simplement :

```python
result_v8 = calibrate(version='v8', df_test=df_test)
```

Tu n'as rien d'autre à modifier.